# Build a USAS dictionary for your language — the recipe as runnable steps

`build_a_language.ipynb` explains what we did and what we measured. This notebook **does
it**: the same six stages, each one a call into `lexicon_pipeline/`, with a *smoke* setting
so the whole thing runs in minutes on a small slice, and a *full* setting for a real build.

| stage | what | needs |
|---|---|---|
| 1 | translate the English USAS word list, word by word | a local open-weights model on an OpenAI-compatible endpoint (Ollama or vLLM) |
| 2 | enrich with words from Wiktionary (the step that helped most) | the free kaikki.org English-Wiktionary extract (~500 MB, once) |
| 3 | enrich with the language's WordNet | `wn` data for your language, if one exists |
| 4 | repair and validate (the release gate UCREL applied) | nothing |
| 5 | build an evaluation reference with a calibrated model committee | API keys **or** local models for three families |
| 6 | score the dictionary | nothing |

The dictionary itself is built with open resources only. Stage 5 is the one place where
strong models are needed; it evaluates, it does not build. Skip it if you have a
human-annotated USAS test set for your language — then go straight to stage 6.

## 1 · Configure

Defaults are a smoke test for **German**: 150 words translated, everything else on the
small side. Set `SMOKE = False` for a real build (the Danish run translated 44k entries in
about two hours on one GPU).

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil
PIPE = Path.cwd() / 'lexicon_pipeline'
PY   = sys.executable

LANG_NAME  = 'German'          # language name in English, as a model would read it
CODE       = 'deu'             # ISO 639-3, used in file names
WIKT       = 'de'              # Wiktionary language code (stage 2)
SPACY      = 'de_core_news_sm' # spaCy model; python -m spacy download de_core_news_sm
WORDNET    = 'omw-de:1.4'      # wn spec for stage 3, or '' to skip
SMOKE      = True              # True: 150 words, minutes. False: the full 54k-entry list

# stage 1 translator: any OpenAI-compatible local endpoint. Ollama serves one at /v1.
OPENAI_BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://localhost:11434/v1')
TRANSLATOR      = os.environ.get('TRANSLATOR_MODEL', 'qwen3:32b')
KAIKKI          = Path(os.environ.get('KAIKKI', 'kaikki-en.jsonl.gz'))   # stage 2 input

WORK = Path('work') / CODE; WORK.mkdir(parents=True, exist_ok=True)
def run(*args):
    print('$', ' '.join(str(a) for a in args)); r = subprocess.run([PY, *map(str, args)], text=True, capture_output=True)
    print(r.stdout[-1500:]); 
    if r.returncode: print(r.stderr[-1500:])
    return r.returncode == 0
print('work dir', WORK.resolve(), '| translator', TRANSLATOR, '@', OPENAI_BASE_URL)

## 2 · Stage 1 — translate the English word list

Each English lemma is translated with its part of speech and its ordered USAS tags kept
as they are: the model translates words, it never invents tags. Checkpointed; re-run to
resume. (Measured: the choice of translator barely matters — a 4-point spread — the next
stage matters more.)

In [ ]:
out1 = PIPE / 'data' / f'semantic_lexicon_{CODE}_{TRANSLATOR.replace(":", "_")}.tsv'
ok = run(PIPE / 'translate_lexicon.py', '--target', LANG_NAME, '--code', CODE, '--model', TRANSLATOR,
         '--base-url', OPENAI_BASE_URL, '--api-key', os.environ.get('OPENAI_API_KEY', 'none'),
         '--workers', '4', *(['--limit', '150'] if SMOKE else []))
cands = sorted((PIPE / 'data').glob(f'semantic_lexicon_{CODE}_*.tsv'))
LEX1 = cands[-1] if cands else None
print('translated lexicon:', LEX1, '|', sum(1 for _ in open(LEX1)) - 1 if LEX1 else 0, 'entries')

## 3 · Stage 2 — add words from Wiktionary

The English Wiktionary lists translations for hundreds of thousands of words. Every
English lemma that has a translation into your language and is in the English USAS list
contributes its tags to the translated word. This was the single most useful step
(+1.4 to +7.6 points). Download the extract once:

```
curl -L -o kaikki-en.jsonl.gz https://kaikki.org/dictionary/English/kaikki.org-dictionary-English.jsonl.gz
```

In [ ]:
LEX2 = WORK / f'semantic_lexicon_{CODE}_wiki.tsv'
if KAIKKI.exists() and LEX1:
    run(PIPE / 'enrich_lexicon_wiktionary.py', '--kaikki', KAIKKI, '--base', LEX1, '--lang-code', WIKT, '--out', LEX2)
else:
    print('kaikki extract not found — skipping stage 2 (set KAIKKI to the downloaded file)'); LEX2 = LEX1

## 4 · Stage 3 — add words from the language's WordNet (optional)

Same idea with a WordNet: English synsets carry USAS tags through the English lexicon;
the target-language words in the same synset inherit them. Small gain, free.

In [ ]:
LEX3 = WORK / f'semantic_lexicon_{CODE}_open.tsv'
if WORDNET and LEX2:
    ok = run(PIPE / 'enrich_lexicon_wordnet.py', '--wordnet', WORDNET, '--base', LEX2, '--out', LEX3)
    if not ok: print('(wordnet step failed — install the wn data: python -m wn download', WORDNET, ') — continuing with stage-2 lexicon'); LEX3 = LEX2
else:
    LEX3 = LEX2

## 5 · Stage 4 — repair and validate

Translation leaves broken records behind: duplicates, multi-word "single words", POS
tokens turned into words, `nan`. `fix_lexicon_errors.py` drives every detected class to
zero and writes a report; it is the gate UCREL's review asked for (`repair_lexicon.py` is the
language-independent version; `fix_lexicon_errors.py` is the original Danish run, kept for the record).

In [ ]:
LEX_FINAL = WORK / f'semantic_lexicon_{CODE}_fixed.tsv'
if LEX3 and Path(LEX3).exists():
    if not run(PIPE / 'repair_lexicon.py', '--lexicon', LEX3, '--out', LEX_FINAL):
        shutil.copy(LEX3, LEX_FINAL)
    print('final lexicon:', LEX_FINAL, '|', sum(1 for _ in open(LEX_FINAL)) - 1, 'entries')

## 6 · Stage 5 — an evaluation reference without human annotators (optional)

If your language has no human-labelled USAS test set, build a *committee reference*: a
short public text is translated by one model family, post-edited by two others, and
tagged by three families. Calibrated on the Finnish and English human gold that ships
with this repo, labels on which **all three families agree** matched the human label
85.5% (Finnish) and 88.2% (English) of the time; where only two agreed, the majority
label was right only 48% and 56% of the time. Score your dictionary on the unanimous
tokens as the primary number. Family separation is the whole point — the translator
family must not be on the committee. This stage needs either API keys for three families
or three local models; it is the only stage where strong models are needed, and it only
*grades*. Cost for the 3,300-word reference text: about $20 at August 2026 list prices.

The first cell below reproduces the calibration card offline from the saved answers (no
API). The second prints the commands for your language; they are printed, not run,
because the keys and models are yours to choose.


In [ ]:
# offline: how trustworthy is the committee? (saved answers vs shipped human gold, no API)
run(PIPE / 'score_calibration_classes.py')


In [ ]:
ref = WORK / f'benedict_{CODE}.txt'
print(f'''# 5a translate the reference text with an INDEPENDENT family (not a committee member)
{PY} {PIPE}/translate_corpus.py --source {PIPE}/data/coffee_eng.txt --language {LANG_NAME} \
    --model openai:{TRANSLATOR} --out {WORK}/coffee_{CODE}_draft.txt
# 5b post-edit with two committee families (fixes applied only where both agree)
{PY} {PIPE}/postedit_corpus.py --source {PIPE}/data/coffee_eng.txt --draft {WORK}/coffee_{CODE}_draft.txt \
    --reviewers openai:gpt-5.6 google:gemini-3.1-pro --out {WORK}/coffee_{CODE}_fixed.txt
# 5c tag with the calibrated committee (strongest first)
{PY} {PIPE}/build_llm_gold.py build --input {WORK}/coffee_{CODE}_fixed.txt --code {CODE} \
    --language-name {LANG_NAME} --spacy-model {SPACY} --out-dir {WORK} \
    --models anthropic:claude-fable-5 openai:gpt-5.6 google:gemini-3.1-pro
# -> {ref}''')

## 7 · Stage 6 — score the dictionary

Exact top-1 accuracy and coverage on the reference: for every word, does the first tag
match? Report both numbers together — a dictionary can gain accuracy by covering less.

In [ ]:
if ref.exists() and LEX_FINAL.exists():
    run(PIPE / 'run_usas_wsd_eval.py', '--gold-file', ref, '--language', CODE, '--language-name', LANG_NAME,
        '--system', 'rule', '--single-lexicon', LEX_FINAL, '--spacy-model', SPACY)
else:
    print('no reference yet (stage 5) — nothing to score. For English/Finnish human gold, download the USAS-WSD test set:')
    print('  https://huggingface.co/datasets/ucrelnlp/USAS-WSD/resolve/main/test/benedict_fin.txt  ->', PIPE / 'data/usas_wsd/')

## Use it in spaCy

```python
from lexicon_pipeline.pymusas_tagger import from_lexicons
nlp = from_lexicons(str(LEX_FINAL), None, SPACY)          # single-word lexicon, no MWE list
doc = nlp("...")
[(t.text, t._.pymusas_tags) for t in doc]
```

The Danish and Dutch dictionaries in `lexicons/` were built exactly this way.